# 0930 21일차

## 1. Bidirectional

RNN을 순방향과 역방향 두 벌로 돌려 결과를 합치는 래퍼 레이어

![같은 입력 x₁, x₂, x₃을 순방향 RNN은 x₁부터, 역방향 RNN은 x₃부터 읽는다. 순방향은 x₃ 칸에서, 역방향은 x₁ 칸에서 끝나고, 두 결과 (N, 10)을 이어 붙여 (N, 20)이 된다](assets/bidirectional.svg)

**필요한 이유**

단방향 RNN은 앞 칸만 보고 다음 칸을 계산함. 양방향으로 두면 칸마다 앞뒤 문맥을 모두 반영할 수 있음

```python
model.add(Bidirectional(SimpleRNN(10), input_shape=(3, 1)))   # (N, 20)
```

1. 순방향 : 1번 칸 → 마지막 칸 순서로 읽음
2. 역방향 : 마지막 칸 → 1번 칸 순서로 읽음
3. 합치기 : 두 결과를 이어 붙임 (`merge_mode='concat'`이 기본값)

- 혼자서는 동작하지 않고 RNN 계열(`SimpleRNN`, `LSTM`, `GRU`)을 감싸서 사용
- 안쪽 RNN을 두 벌로 복사해서 계산하므로 출력은 `units`의 2배, 파라미터도 2배
- 입력 전체가 주어진 데이터에 씀. 한 칸씩 들어오면서 바로 답해야 하면 역방향이 읽을 뒤 칸이 없음

### 1-1. 파라미터 개수

한 방향의 개수를 구해서 2배

| | 계산 | Param # |
|---|---|---|
| `SimpleRNN(10)` 한 방향 | `10 × (1 + 10 + 1)` | 120 |
| `Bidirectional` | `120 × 2` | 240 |

- 한 방향은 day18 §3-4의 공식 그대로
- 출력 `(N, 20)`을 받는 다음 `Dense`는 입력이 20개인 것으로 계산

#### cf) input_shape 위치

`input_shape`는 안쪽 RNN이 아니라 바깥 `Bidirectional`에 넣음

- `Sequential`은 맨 앞 층의 `input_shape`로 입력 모양을 정하는데, 맨 앞 층은 `Bidirectional`임
- 안쪽에 넣으면 쓰이지 않아 `summary()`의 Output Shape가 `?`, Param #가 `0 (unbuilt)`로 나옴